# Financial Data Analysis with Python

## Learning objectives

By the end of this notebook, you should be able to:

- Build a structured market-data analysis workflow in Python.
- Clean and standardize downloaded price data for analysis.
- Engineer key return and risk features used in finance.
- Communicate insights with clear, professional charts.

## Where this notebook fits in the course

This is the first notebook of the mini-course. It establishes the data workflow discipline that we will reuse in Notebook 2 (LBO modeling) and Notebook 3 (algorithmic trading).

## Notebook narrative

We follow a practical analyst sequence:

1. Define the financial question and collect relevant data.
2. Audit data quality and create a reliable analysis base.
3. Engineer interpretable features (returns, trend, risk).
4. Visualize results and translate them into financial interpretation.


## Libraries used and why

- `pandas`: time-series data manipulation and table organization.
- `numpy`: numerical operations and log-return calculations.
- `matplotlib` and `seaborn`: readable, publication-style charts.
- `yfinance`: programmatic access to liquid market data.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook", palette="deep")
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelsize"] = 11
pd.set_option("display.float_format", "{:.4f}".format)


## 1. Data collection

**Financial problem:** compare performance and risk behavior across liquid U.S. assets.

We use:

- `AAPL` and `MSFT` as large-cap equities,
- `SPY` as a broad market benchmark proxy.


In [ ]:
tickers = ["AAPL", "MSFT", "SPY"]
start_date = "2018-01-01"
end_date = pd.Timestamp.today().strftime("%Y-%m-%d")

raw = yf.download(
    tickers=tickers,
    start=start_date,
    end=end_date,
    auto_adjust=False,
    progress=False,
)

if raw.empty:
    raise ValueError("No data returned by yfinance. Verify internet access and ticker symbols.")

raw.tail()


### Section transition

With raw data collected, the next step is to verify quality and choose a consistent price definition before building any indicators.


## 2. Data inspection and cleaning

We inspect available fields and missing values, then standardize the working price table.

For consistency, we prioritize **Adjusted Close** (when available), which reflects splits and dividends.


In [ ]:
print("Top-level columns:", list(raw.columns.levels[0]))
print("\nMissing observations by field (first rows):")
print(raw.isna().sum().head())

price_col = "Adj Close" if "Adj Close" in raw.columns.levels[0] else "Close"
prices = raw[price_col].copy().sort_index()
prices = prices.dropna(how="all").ffill().dropna()

print(f"\nSelected price column: {price_col}")
print(f"Date range: {prices.index.min().date()} to {prices.index.max().date()}")
prices.head()


### Discussion

A reliable analysis starts with explicit data choices. Defining one consistent pricing convention avoids silent inconsistencies in returns and risk metrics.

### Section transition

After data quality checks, we can engineer features that map directly to common finance questions: performance, volatility, and trend.


## 3. Feature engineering

We compute a compact set of high-value features:

- Daily simple returns,
- Daily log returns,
- 21-day rolling annualized volatility,
- 50-day and 200-day moving averages.


In [ ]:
returns = prices.pct_change().dropna()
log_returns = np.log(prices / prices.shift(1)).dropna()
rolling_vol_21d = returns.rolling(21).std() * np.sqrt(252)
cumulative_returns = (1 + returns).cumprod()

focus = "SPY"
trend = pd.DataFrame(index=prices.index)
trend["price"] = prices[focus]
trend["ma_50"] = prices[focus].rolling(50).mean()
trend["ma_200"] = prices[focus].rolling(200).mean()

feature_table = pd.DataFrame({
    "daily_return": returns[focus],
    "log_return": log_returns[focus],
    "rolling_vol_21d": rolling_vol_21d[focus],
    "ma_50": trend["ma_50"],
    "ma_200": trend["ma_200"],
}).dropna()

feature_table.tail()


### Discussion

These features are intentionally simple and interpretable. They are sufficient to discuss market behavior without obscuring the workflow with unnecessary technical complexity.

### Section transition

We now move from feature construction to visualization, where the goal is to communicate financial interpretation clearly.


## 4. Visual analytics

The next charts summarize trend, relative performance, risk dynamics, and return distribution.


In [ ]:
fig, ax = plt.subplots()
ax.plot(trend.index, trend["price"], label=f"{focus} Price", color="#1f77b4", linewidth=1.8)
ax.plot(trend.index, trend["ma_50"], label="50-day MA", color="#ff7f0e", linewidth=1.4)
ax.plot(trend.index, trend["ma_200"], label="200-day MA", color="#2ca02c", linewidth=1.4)
ax.set_title(f"{focus}: Price and Moving Averages")
ax.set_xlabel("Date")
ax.set_ylabel("Price (USD)")
ax.legend(frameon=True)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots()
for t in tickers:
    ax.plot(cumulative_returns.index, cumulative_returns[t], label=t, linewidth=1.6)
ax.set_title("Cumulative Returns (Base = 1)")
ax.set_xlabel("Date")
ax.set_ylabel("Growth of $1")
ax.legend(frameon=True)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots()
ax.plot(rolling_vol_21d.index, rolling_vol_21d[focus], color="#d62728", linewidth=1.6)
ax.set_title(f"{focus}: 21-Day Rolling Annualized Volatility")
ax.set_xlabel("Date")
ax.set_ylabel("Volatility")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(returns[focus], bins=60, kde=True, color="#4c72b0", alpha=0.85, ax=ax)
ax.axvline(returns[focus].mean(), color="black", linestyle="--", linewidth=1.2, label="Mean")
ax.set_title(f"{focus}: Distribution of Daily Returns")
ax.set_xlabel("Daily Return")
ax.set_ylabel("Frequency")
ax.legend(frameon=True)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 5. Interpretation of results

- Moving averages provide a simple trend regime view and help contextualize market phases.
- Cumulative returns show that path and drawdown history matter, not only final performance.
- Rolling volatility highlights changing risk regimes over time.
- Return distributions are noisy and asymmetric enough to remind us that normality assumptions can be fragile.

### Discussion

A professional notebook should make interpretation explicit. Visual output is only useful when tied back to the financial question.


## Key takeaways

- A robust financial analysis notebook follows a clear pipeline: **collect -> clean -> engineer -> visualize -> interpret**.
- Transparent data conventions (for example, adjusted prices) are essential for reproducibility.
- Compact, high-quality visuals support clearer analytical communication.

## Bridge to Notebook 2

In this notebook, we built discipline around data preparation and interpretation. The next notebook applies the same discipline to corporate finance by structuring an LBO model where assumptions, projection logic, and value drivers are explicitly linked.
